# Competitor Monitor

Feeds the "Competitor activities" part of the Market Intelligence dashboard section.

Important design note: this does **not** use a separate competitor database.
It searches the same `clean_data.json` knowledge repository used everywhere
else in the project, just filtered by competitor name instead of by
opportunity/risk framing. Since our data collection filters for articles
that mention SAP, competitor coverage here is limited to articles where a
competitor is mentioned *alongside* SAP (e.g. comparison or market-share
articles) - not standalone competitor-only news.


In [ ]:
import json
import os
import re
import time

import pandas as pd
import requests
from pathlib import Path

cwd = Path.cwd()
DATA_DIR = next((p for p in [cwd / "notebook" / "data", cwd.parent / "notebook" / "data"] if p.exists()), cwd)

CLEAN_DATA_PATH = DATA_DIR / "clean_data.json"
OUT_PATH        = DATA_DIR / "competitor_activity.json"

OLLAMA_HOST  = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "phi4-mini")
TIMEOUT      = 300

print("data dir:", DATA_DIR)


In [ ]:
df = pd.read_json(CLEAN_DATA_PATH)
print("articles loaded:", len(df))


In [ ]:
def ask_llm(prompt):
    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    return resp.json()["response"]


def warmup():
    print("warming up model...")
    start = time.time()
    ask_llm("hi")
    print("warmup done in", round(time.time() - start, 1), "sec")


warmup()


## Find competitor mentions

Word-boundary match so "Infor" doesn't accidentally match inside words like
"reinforcing". Counts come from the cleaned corpus, not assumptions.


In [ ]:
COMPETITORS = ["Oracle", "Workday", "Salesforce", "Microsoft", "Infor", "IFS", "ServiceNow", "Epicor"]

def find_mentions(name):
    pattern = r"\b" + re.escape(name) + r"\b"
    mask = df["text"].str.contains(pattern, case=False, regex=True, na=False)
    return df[mask]


for name in COMPETITORS:
    rows = find_mentions(name)
    print(name, "-", len(rows), "articles")


## Summarize each competitor's activity

For competitors with at least one mention, build context from the matching
articles and ask the local model to summarize what they're doing relative
to SAP. Competitors with zero mentions are skipped but still listed so the
dashboard can show "no recent activity detected."


In [ ]:
def summarize_competitor(name, rows):
    context = "\n\n".join(rows["title"] + ". " + rows["clean_text"].str.slice(0, 800))

    prompt = f"""You are a market intelligence analyst for SAP.

Below are news snippets that mention {name}. Summarize in 2-3 sentences what
{name} is doing, and how it relates to SAP if mentioned.

Return ONLY a JSON object, no extra text, no markdown, in this format:

{{
  "competitor": "{name}",
  "summary": "2-3 sentence summary",
  "mention_count": {len(rows)}
}}

News snippets:
{context}
"""
    raw = ask_llm(prompt)
    return raw


def parse_json_obj(raw_text):
    text = raw_text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text.replace("json", "", 1).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("could not parse JSON, here is the raw text:")
        print(text)
        return None


In [ ]:
results = []

for name in COMPETITORS:
    rows = find_mentions(name)

    if len(rows) == 0:
        results.append({
            "competitor": name,
            "summary": "No recent activity detected in the current corpus.",
            "mention_count": 0,
        })
        continue

    raw_output = summarize_competitor(name, rows)
    parsed = parse_json_obj(raw_output)

    if parsed is None:
        parsed = {"competitor": name, "summary": raw_output, "mention_count": len(rows)}

    results.append(parsed)
    print(name, "done")

for r in results:
    print("-", r.get("competitor"), "|", r.get("mention_count"), "mentions")


## Save results for the dashboard

In [ ]:
with open(OUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print("saved to", OUT_PATH)
